In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
os.chdir('/content/drive/My Drive/colab/CSC6502 - Computer Vision/cv_practical_work/csc6502-prac6-CHIPOMPO')
CWD_PATH = os.getcwd()
print(CWD_PATH)

/content/drive/My Drive/colab/CSC6502 - Computer Vision/cv_practical_work/csc6502-prac6-CHIPOMPO


In [ ]:
import cv2
import numpy as np
from pathlib import Path
from google.colab import files
from IPython.display import Video, display
#%pip uninstall -y opencv-python opencv-contrib-python opencv-python-headless opencv-contrib-python-headless
#%pip install opencv-python

## Task 4. Augmented reality: a cube on the chessboard

In [ ]:

CALIBRATION_FILE = "images/calibration.npz"
INPUT_VIDEO = "videos/chessboard_pan.mp4"
OUTPUT_VIDEO = "videos/chessboard_ar_output.mp4"

# Number of INNER corners: (columns, rows)
CHESSBOARD_SIZE = (7, 7)

# Use the same square-size unit consistently throughout.
SQUARE_SIZE = 1.0
CUBE_HEIGHT = 3.0

print("OpenCV version:", cv2.__version__)

OpenCV version: 5.0.0


In [ ]:
def load_calibration(calibration_file):
    """Load the camera matrix and distortion coefficients from an NPZ file."""

    if not Path(calibration_file).exists():
        raise FileNotFoundError(
            f"Calibration file not found: {calibration_file}"
        )

    with np.load(calibration_file) as data:
        print("Available calibration keys:", data.files)

        matrix_keys = ["camera_matrix", "cameraMatrix", "mtx", "K"]
        distortion_keys = [
            "dist_coeffs", "distCoeffs", "dist",
            "distortion", "distortion_coefficients"
        ]

        camera_matrix = next(
            (data[k] for k in matrix_keys if k in data.files), None
        )
        dist_coeffs = next(
            (data[k] for k in distortion_keys if k in data.files), None
        )

        if camera_matrix is None or dist_coeffs is None:
            raise KeyError(
                "Could not identify calibration arrays. "
                "Inspect the available keys and update the key lists."
            )

    return (
        np.asarray(camera_matrix, dtype=np.float64),
        np.asarray(dist_coeffs, dtype=np.float64)
    )


camera_matrix, dist_coeffs = load_calibration(CALIBRATION_FILE)

print("\nCamera matrix:\n", camera_matrix)
print("\nDistortion coefficients:\n", dist_coeffs)

Available calibration keys: ['K', 'dist', 'image_size', 'fx', 'fy', 'cx', 'cy', 'focal_mm', 'sensor_width_mm', 'rms']

Camera matrix:
 [[2.82819686e+03 0.00000000e+00 1.53004752e+03]
 [0.00000000e+00 2.83122865e+03 2.00296263e+03]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]]

Distortion coefficients:
 [[ 0.13209791 -0.44242192  0.0006959   0.00140488  0.40070856]]


In [ ]:
def create_board_points(board_size, square_size):
    """Create 3D coordinates for the chessboard's inner corners."""

    cols, rows = board_size

    points = np.zeros((rows * cols, 3), dtype=np.float32)
    points[:, :2] = (
        np.mgrid[0:cols, 0:rows].T.reshape(-1, 2) * square_size
    )

    return points


def create_cube_points(square_size, cube_height):
    """Create the eight vertices of a cube resting on the board."""

    s = 2.0 * square_size
    h = cube_height * square_size

    return np.float32([
        [0, 0, 0],
        [s, 0, 0],
        [s, s, 0],
        [0, s, 0],
        [0, 0, h],
        [s, 0, h],
        [s, s, h],
        [0, s, h]
    ])


def create_axis_points(square_size):
    """Create the origin and endpoints of the three coordinate axes."""

    length = 3.0 * square_size

    return np.float32([
        [0, 0, 0],
        [length, 0, 0],  # X
        [0, length, 0],  # Y
        [0, 0, length]   # Z
    ])


board_points = create_board_points(CHESSBOARD_SIZE, SQUARE_SIZE)
cube_points = create_cube_points(SQUARE_SIZE, CUBE_HEIGHT)
axis_points = create_axis_points(SQUARE_SIZE)

cube_edges = [
    (0, 1), (1, 2), (2, 3), (3, 0),
    (4, 5), (5, 6), (6, 7), (7, 4),
    (0, 4), (1, 5), (2, 6), (3, 7)
]

print("Chessboard points:", board_points.shape)
print("Cube vertices:", cube_points.shape)
print("Axis points:", axis_points.shape)

Chessboard points: (49, 3)
Cube vertices: (8, 3)
Axis points: (4, 3)


In [ ]:
def detect_chessboard_corners(frame, board_size):
    """Detect and refine chessboard corners in a video frame."""

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    found, corners = cv2.findChessboardCorners(
        gray,
        board_size,
        flags=(
            cv2.CALIB_CB_ADAPTIVE_THRESH
            + cv2.CALIB_CB_NORMALIZE_IMAGE
        )
    )

    if not found:
        return False, None

    criteria = (
        cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER,
        30,
        0.001
    )

    refined_corners = cv2.cornerSubPix(
        gray, corners, (11, 11), (-1, -1), criteria
    )

    return True, refined_corners

In [ ]:
def estimate_board_pose(corners, board_points, camera_matrix, dist_coeffs):
    """Estimate the board's pose relative to the camera."""

    success, rvec, tvec = cv2.solvePnP(
        board_points,
        corners,
        camera_matrix,
        dist_coeffs,
        flags=cv2.SOLVEPNP_ITERATIVE
    )

    if not success:
        return None, None

    return rvec, tvec


def draw_projected_cube(frame, cube_points, rvec, tvec,
                        camera_matrix, dist_coeffs):
    """Project and draw the twelve edges of a virtual cube."""

    projected, _ = cv2.projectPoints(
        cube_points, rvec, tvec, camera_matrix, dist_coeffs
    )

    points = np.round(projected.reshape(-1, 2)).astype(int)

    for start, end in cube_edges:
        cv2.line(
            frame,
            tuple(points[start]),
            tuple(points[end]),
            (0, 255, 0),
            2,
            cv2.LINE_AA
        )

    return frame


def draw_projected_axes(frame, axis_points, rvec, tvec,
                        camera_matrix, dist_coeffs):
    """Draw X in red, Y in green and Z in blue."""

    projected, _ = cv2.projectPoints(
        axis_points, rvec, tvec, camera_matrix, dist_coeffs
    )

    points = np.round(projected.reshape(-1, 2)).astype(int)
    origin = tuple(points[0])

    axis_specs = [
        (1, (0, 0, 255), "X"),
        (2, (0, 255, 0), "Y"),
        (3, (255, 0, 0), "Z")
    ]

    for index, color, label in axis_specs:
        endpoint = tuple(points[index])

        cv2.line(
            frame, origin, endpoint, color, 3, cv2.LINE_AA
        )
        cv2.putText(
            frame, label, endpoint,
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2
        )

    return frame

In [ ]:
def process_video(input_video, output_video,
                  board_size, board_points, cube_points, axis_points,
                  camera_matrix, dist_coeffs):
    """Detect the board, estimate its pose, draw AR objects and save video."""

    if not Path(input_video).exists():
        raise FileNotFoundError(f"Input video not found: {input_video}")

    cap = cv2.VideoCapture(input_video)

    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {input_video}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    if not np.isfinite(fps) or fps <= 0:
        fps = 30.0

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    Path(output_video).parent.mkdir(parents=True, exist_ok=True)

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(
        output_video, fourcc, fps, (width, height)
    )

    if not writer.isOpened():
        cap.release()
        raise RuntimeError("Could not initialise the output video writer.")

    frame_count = 0
    detected_count = 0

    try:
        while True:
            ok, frame = cap.read()

            if not ok:
                break

            frame_count += 1

            found, corners = detect_chessboard_corners(
                frame, board_size
            )

            if found:
                rvec, tvec = estimate_board_pose(
                    corners, board_points,
                    camera_matrix, dist_coeffs
                )

                if rvec is not None:
                    detected_count += 1

                    frame = draw_projected_cube(
                        frame, cube_points, rvec, tvec,
                        camera_matrix, dist_coeffs
                    )

                    frame = draw_projected_axes(
                        frame, axis_points, rvec, tvec,
                        camera_matrix, dist_coeffs
                    )

                    cv2.drawChessboardCorners(
                        frame, board_size, corners, True
                    )
                else:
                    cv2.putText(
                        frame, "Pose estimation failed", (20, 35),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8,
                        (0, 0, 255), 2
                    )
            else:
                cv2.putText(
                    frame, "Chessboard not detected", (20, 35),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8,
                    (0, 0, 255), 2
                )

            writer.write(frame)

            if frame_count % 30 == 0:
                print(
                    f"Processed {frame_count} frames; "
                    f"successful pose estimates: {detected_count}"
                )

    finally:
        cap.release()
        writer.release()

    print("\nProcessing complete.")
    print("Total frames:", frame_count)
    print("Frames with successful pose estimates:", detected_count)
    print("Output video:", output_video)

    if detected_count == 0:
        print(
            "WARNING: No successful detections. Check board dimensions, "
            "video path, and calibration compatibility."
        )

    return frame_count, detected_count


frame_count, detected_count = process_video(
    INPUT_VIDEO,
    OUTPUT_VIDEO,
    CHESSBOARD_SIZE,
    board_points,
    cube_points,
    axis_points,
    camera_matrix,
    dist_coeffs
)

Processed 30 frames; successful pose estimates: 30
Processed 60 frames; successful pose estimates: 60
Processed 90 frames; successful pose estimates: 90
Processed 120 frames; successful pose estimates: 120
Processed 150 frames; successful pose estimates: 150
Processed 180 frames; successful pose estimates: 180
Processed 210 frames; successful pose estimates: 210
Processed 240 frames; successful pose estimates: 240
Processed 270 frames; successful pose estimates: 270
Processed 300 frames; successful pose estimates: 300
Processed 330 frames; successful pose estimates: 330
Processed 360 frames; successful pose estimates: 360
Processed 390 frames; successful pose estimates: 390
Processed 420 frames; successful pose estimates: 420
Processed 450 frames; successful pose estimates: 450
Processed 480 frames; successful pose estimates: 479
Processed 510 frames; successful pose estimates: 509
Processed 540 frames; successful pose estimates: 539

Processing complete.
Total frames: 555
Frames with 

The augmented reality implementation uses `cv2.solvePnP()` to estimate the position and orientation ot the chessboard relative to the camera. It does this by matching known three-dimensional chessboard corner coordinates with their corresponding two-dimensional positions detected in each video frame, using the camera intrinsic matrix and distortion coefficients from Task 3. Camera calibration was necessary before pose estimation because it provides the intrinsic parameters and lens-distortion information required for accurate geometric projection resulting into the projected cube appearing anchored to the chessboard as the camera moves.